# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My lane: Refresh / Content Opportunity Scoring

I'm choosing Lane 2 because I've already worked inside this exact problem during Week 1
(Notebooks 1 and 2, both part of the same first assignment): comparing a hand-written
refresh rule against a learned model on Precision@50. The starter pipeline's own results
show a clear, real gap between the baseline rule and a trained model, which is strong
early evidence this lane has enough signal to be worth building on for the next 7 weeks.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The question

**Research question:** Which content pages should be prioritized for review first —
for refresh, expansion, or monitoring — given limited reviewer capacity?

**Unit of analysis:** one content page (`content_id`), scored at a point in time.

**Output:** a ranked queue of pages, each with a score and a reason code explaining
why it was flagged (e.g. stale + visible, declining with demand, thin content with
traffic).

**The decision this improves:** which pages a content reviewer looks at first, out
of a much larger inventory, when they only have time to review a limited number
per cycle.

**The action someone takes:** a reviewer opens a flagged page, checks the reason
code, and decides whether to refresh, expand, consolidate, or leave it — this
model doesn't take the action itself, it just orders the queue.

**Cost of a wrong call:**
- *False positive* (flagged as worth reviewing, but it wasn't): wastes a reviewer's
  limited time — the real cost of any ranking system with fixed capacity.
- *False negative* (a genuinely declining page never surfaces in the top-K):
  a page keeps losing visibility/traffic with no one aware, until it's caught by
  chance or a later scoring pass.

**Why data/ML can help at all:** with hundreds of thousands of pages and a small
review team, a fixed hand-written rule is fast but rigid — it can't weigh multiple
weak signals together the way a learned model can. The starter data already shows
this gap is real, not theoretical (see numbers below).

**One-paragraph frame (per framing-ml-problems):**

> For a content reviewer with limited weekly capacity, deciding which pages to check first,
> we will build a ranked priority queue from the starter content-performance dataset,
> scoring pages by likelihood of being in genuine decline, measured by Precision@50. A wrong
> call costs either wasted reviewer time (false positive) or a real decline going unnoticed
> (false negative). A plain rule isn't enough because relevant signals — staleness, visibility,
> position, thin content — interact in ways a single hand-written rule can't weigh together;
> the starter data already shows this (baseline 0.240 vs. random forest 0.740 Precision@50).
> We will claim only observed / directional / decision-support results — not that a refresh
> will cause recovery, and not that the current label is a true future outcome rather than a
> current-window proxy.

**Task type (per framing-ml-problems):** Ranking / scoring — target is a priority score,
metric is Precision@K.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os
os.chdir("/content")
print(os.getcwd())

/content


In [2]:
import os, sys, subprocess

os.chdir("/content")

REPO_URL = "https://github.com/Santosh-S321/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


In [3]:
import sys, subprocess
subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_all.py'], returncode=0)

In [4]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Baseline rule Precision@50: {base:.3f}  (~{round(base*50)} of top 50 correct)")
print(f"Random forest Precision@50: {rf:.3f}  (~{round(rf*50)} of top 50 correct)")
print(f"Lift: {rf/base:.1f}x")

Baseline rule Precision@50: 0.240  (~12 of top 50 correct)
Random forest Precision@50: 0.740  (~37 of top 50 correct)
Lift: 3.1x


**Why this lane looks worth 7 weeks:**

- Baseline rule: Precision@50 = 0.240 (~12 of the top 50 flagged pages actually declining)
- Random forest: Precision@50 = 0.740 (~37 of the top 50 correct) — roughly a 3x lift
  over the hand-written rule, on the 30,000-row starter sample with client-holdout validation

- When I re-tested the hand-rule-vs-tree comparison on a proper held-out split myself
  (Notebook 2, Week 2), the hand rule's precision dropped as expected under held-out
  evaluation (0.900→0.700 at K=20), showing the in-sample numbers were somewhat
  optimistic — exactly the kind of gap this lane is built to catch and correct for.

These numbers are from the anonymized starter slice only, not the full warehouse —
but they're real, measured, and reproducible from `outputs/model_results.json`.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## What I can and can't claim

**I can claim:**
- The learned model outperforms the hand-written rule on Precision@50, on this
  starter sample, under client-holdout validation.
- This is decision-support: it ranks candidates for a human to review, not an
  automated fix.

**I can't claim:**
- That refreshing a flagged page will cause traffic to recover — that needs a
  causal experiment, not this ranking model.
- That the current starter label (`trend_direction == "down"`) is the ideal
  target — it's a proxy based on the current window, not a future outcome.
  A stronger capstone version should move toward a future-window label:
  prior 90 days of features → decline or recovery over the next 30 days.
- Anything about Google's ranking algorithm — this only measures FlyRank's own
  observed search and engagement data.

**Data rule going forward (per flyrank-data):** `trend_direction` and `trend_pct`
are the source of the current label — they can never be used as model features
in later weeks, only as the label itself. Also noting for later: `avg_position = 0`
means "no data," not rank zero; rate columns (`ctr`, `engagement_rate`, etc.) are
×100 percentages, not raw fractions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.